# Sematic Chunking

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt")

def semantic_chunking(text: str, max_chunk_size: int):
    """
    Splits text into semantically coherent chunks based on sentence boundaries.

    This function uses sentence tokenization to ensure chunks do not break 
    mid-sentence. Each chunk's size is limited to `max_chunk_size` characters.
    If adding a new sentence would exceed the maximum chunk size, the current 
    chunk is saved, and a new chunk is started.

    Parameters:
    --------
    text : str
        The input text to be segmented into semantic chunks.
    max_chunk_size : int
        The maximum number of characters allowed in a single chunk.

    Returns:
    -------
    list[str]
        A list of text chunks, each respecting sentence boundaries.
    """
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chunk_size:
            current_chunk += sentence + " "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    
    if current_chunk:  # Add any remaining text
        chunks.append(current_chunk.strip())
    
    return chunks

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


In [ ]:
semantic_chunking()

# Content-Aware Chunking (Mardown format)

In [ ]:
import os

from langchain_community.document_loaders import TextLoader

DATA_PATH = "data"

document = TextLoader(os.path.join(DATA_PATH, "nextjs_intro.md"), encoding='utf-8').load()

document

[Document(metadata={'source': 'data\\nextjs_intro.md'}, page_content='---\ntitle: How to set up a Next.js project\nnav_title: Installation\ndescription: Learn how to create a new Next.js application with `create-next-app`, and set up TypeScript, ESLint, and Module Path Aliases.\n---\n\n{/* The content of this doc is shared between the app and pages router. You can use the `<PagesOnly>Content</PagesOnly>` component to add content that is specific to the Pages Router. Any shared content should not be wrapped in a component. */}\n\n## System requirements\n\n- [Node.js 18.18](https://nodejs.org/) or later.\n- macOS, Windows (including WSL), and Linux are supported.\n\n## Automatic installation\n\nWe recommend starting a new Next.js app using [`create-next-app`](/docs/app/api-reference/cli/create-next-app), which sets up everything automatically for you. To create a project, run:\n\n```bash filename="Terminal"\nnpx create-next-app@latest\n```\n\nOn installation, you\'ll see the following pr

Split by Markdown format

In [2]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_document: str = document[0].page_content # Document (string type) in markdown format

headers_to_split_on = [
    ("#", "Header 1"),    # Level 1 headers (e.g., ## Book 1)
    ("##", "Header 2"),   # Level 2 headers (e.g., ## Chapter 1)
    ("###", "Header 3"),    # Level 3 headers (e.g., ### Chapter 1.1)
    ("####", "Header 4"),   # Level 4 headers (e.g., #### Chapter 1.1.1)
    ("#####", "Header 5"),   # Level 5 headers (e.g., ##### Chapter 1.1.1.1)
]

# MD splits
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False, return_each_line=True
)
md_header_splits = markdown_splitter.split_text(markdown_document)

md_header_splits

[Document(metadata={}, page_content='---\ntitle: How to set up a Next.js project\nnav_title: Installation\ndescription: Learn how to create a new Next.js application with `create-next-app`, and set up TypeScript, ESLint, and Module Path Aliases.\n---'),
 Document(metadata={}, page_content='{/* The content of this doc is shared between the app and pages router. You can use the `<PagesOnly>Content</PagesOnly>` component to add content that is specific to the Pages Router. Any shared content should not be wrapped in a component. */}'),
 Document(metadata={'Header 2': 'System requirements'}, page_content='## System requirements'),
 Document(metadata={'Header 2': 'System requirements'}, page_content='- [Node.js 18.18](https://nodejs.org/) or later.\n- macOS, Windows (including WSL), and Linux are supported.'),
 Document(metadata={'Header 2': 'Automatic installation'}, page_content='## Automatic installation'),
 Document(metadata={'Header 2': 'Automatic installation'}, page_content='We recom

Label each document as `text` or `code`

In [3]:
from test_util import TextAndCodeLabeler

text_and_code_splitter = TextAndCodeLabeler()
text_and_code_splits = text_and_code_splitter.label_docs(md_header_splits)

text_and_code_splits

[RAGDocument(metadata={'type': 'text'}, page_content='---\ntitle: How to set up a Next.js project\nnav_title: Installation\ndescription: Learn how to create a new Next.js application with `create-next-app`, and set up TypeScript, ESLint, and Module Path Aliases.\n---'),
 RAGDocument(metadata={'type': 'text'}, page_content='{/* The content of this doc is shared between the app and pages router. You can use the `<PagesOnly>Content</PagesOnly>` component to add content that is specific to the Pages Router. Any shared content should not be wrapped in a component. */}'),
 RAGDocument(metadata={'Header 2': 'System requirements', 'type': 'text'}, page_content='## System requirements'),
 RAGDocument(metadata={'Header 2': 'System requirements', 'type': 'text'}, page_content='- [Node.js 18.18](https://nodejs.org/) or later.\n- macOS, Windows (including WSL), and Linux are supported.'),
 RAGDocument(metadata={'Header 2': 'Automatic installation', 'type': 'text'}, page_content='## Automatic instal

- Using Fixed-sized approach

In [4]:
from test_util import FixedSizeTextSplitter

text_splitter = FixedSizeTextSplitter(chunk_size=100, delete_whitespace=True)
splits = text_splitter.split_docs(md_header_splits)

splits


[RAGDocument(metadata={'type': 'text'}, page_content='---\ntitle: How to set up a Next.js project\nnav_title: Installation\ndescription: Learn how to create '),
 RAGDocument(metadata={'type': 'text'}, page_content='a new Next.js application with `create-next-app`, and set up TypeScript, ESLint, and Module Path Ali'),
 RAGDocument(metadata={'type': 'text'}, page_content='ases.\n---'),
 RAGDocument(metadata={'type': 'text'}, page_content='{/* The content of this doc is shared between the app and pages router. You can use the `<PagesOnly>'),
 RAGDocument(metadata={'type': 'text'}, page_content='Content</PagesOnly>` component to add content that is specific to the Pages Router. Any shared conte'),
 RAGDocument(metadata={'type': 'text'}, page_content='nt should not be wrapped in a component. */}'),
 RAGDocument(metadata={'Header 2': 'System requirements', 'type': 'text'}, page_content='## System requirements'),
 RAGDocument(metadata={'Header 2': 'System requirements', 'type': 'text'}, page_

- Using Sliding Window approach

In [5]:
from test_util import SlidingWindowTextSplitter

text_splitter = SlidingWindowTextSplitter(chunk_window_size=100, stride=20, delete_whitespace=True)
splits = text_splitter.split_docs(md_header_splits)

splits

[RAGDocument(metadata={'type': 'text'}, page_content='---\ntitle: How to set up a Next.js project\nnav_title: Installation\ndescription: Learn how to create '),
 RAGDocument(metadata={'type': 'text'}, page_content='t up a Next.js project\nnav_title: Installation\ndescription: Learn how to create a new Next.js applic'),
 RAGDocument(metadata={'type': 'text'}, page_content='ct\nnav_title: Installation\ndescription: Learn how to create a new Next.js application with `create-n'),
 RAGDocument(metadata={'type': 'text'}, page_content='lation\ndescription: Learn how to create a new Next.js application with `create-next-app`, and set up'),
 RAGDocument(metadata={'type': 'text'}, page_content='Learn how to create a new Next.js application with `create-next-app`, and set up TypeScript, ESLint,'),
 RAGDocument(metadata={'type': 'text'}, page_content='a new Next.js application with `create-next-app`, and set up TypeScript, ESLint, and Module Path Ali'),
 RAGDocument(metadata={'type': 'text'}, page

- Using Structured-Aware approach (by sentence)

In [6]:
from test_util import SentenceTextSplitter

text_splitter = SentenceTextSplitter()
splits = text_splitter.split_docs(md_header_splits)

splits

[RAGDocument(metadata={'type': 'text'}, page_content='--- title: How to set up a Next.js project nav_title: Installation description: Learn how to create a new Next.js application with `create-next-app`, and set up TypeScript, ESLint, and Module Path Aliases'),
 RAGDocument(metadata={'type': 'text'}, page_content='---'),
 RAGDocument(metadata={'type': 'text'}, page_content='{/* The content of this doc is shared between the app and pages router'),
 RAGDocument(metadata={'type': 'text'}, page_content='You can use the `<PagesOnly>Content</PagesOnly>` component to add content that is specific to the Pages Router'),
 RAGDocument(metadata={'type': 'text'}, page_content='Any shared content should not be wrapped in a component'),
 RAGDocument(metadata={'type': 'text'}, page_content='*/}'),
 RAGDocument(metadata={'Header 2': 'System requirements', 'type': 'text'}, page_content='## System requirements'),
 RAGDocument(metadata={'Header 2': 'System requirements', 'type': 'text'}, page_content='- [

# Sentence Window

In [7]:
# import nltk
# from nltk.tokenize import sent_tokenize

# # Ensure you have downloaded the necessary NLTK data files
# nltk.download('punkt')

# def sentence_window(text, keyword, window_size=2):
#     # Tokenize the text into sentences
#     sentences = sent_tokenize(text)
    
#     # Create a list to hold the sentences in the window
#     window_sentences = []
    
#     # Iterate through the sentences to find the keyword
#     for index, sentence in enumerate(sentences):
#         if keyword in sentence:
#             # Get the start and end indices for the window
#             start_index = max(index - window_size, 0)
#             end_index = min(index + window_size + 1, len(sentences))
            
#             # Add the sentences in the window to the list
#             window_sentences.extend(sentences[start_index:end_index])
    
#     return window_sentences

# # Example usage
# text = ("The quick brown fox jumps over the lazy dog. "
#         "This is a sample sentence that includes the keyword. "
#         "The lazy dog then barks loudly. "
#         "The fox runs away quickly.")
# keyword = "keyword"

# result = sentence_window(text, keyword, window_size=1)
# for sentence in result:
#     print(sentence)